# Prep augmented per-class datasets (tabDDPM + CTGAN)

This notebook:
- reads each per-class synthetic CSV from `5G-NIDD_Aug_tabDDPM/<AttackType>/synthetic_tabddpm_unconditional.csv`
  and/or `5G-NIDD_Aug_CTGAN/<AttackType>/synthetic_ctgan_unconditional.csv`
- uses the per-attack train file `train_input_with_label.csv` inside the same folder (skips `Balanced`)
- also reads the per-attack test file `test_input_with_label.csv` inside the same folder
- adds missing label columns to synthetic data using the corresponding train file
- writes per-class augmented CSVs to `Datasets/augmented_per_class_datasets/<model>/mixed_augmented_<AttackType>.csv`
- writes a combined shuffled train+synthetic CSV to `Datasets/augmented_per_class_datasets/<model>/mixed_augmented_per_class_merged.csv`
- writes a combined shuffled real test CSV per model to `Datasets/augmented_per_class_datasets/<model>/test_input_with_label_merged.csv`


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd


## Config


In [ ]:
MODEL_CONFIGS = {
    "tabddpm": {
        "root": Path("5G-NIDD_Aug_tabDDPM"),
        "synthetic_filename": "synthetic_tabddpm_unconditional.csv",
    },
    "ctgan": {
        "root": Path("5G-NIDD_Aug_CTGAN"),
        "synthetic_filename": "synthetic_ctgan_unconditional.csv",
    },
}

MODELS_TO_RUN = ["tabddpm", "ctgan"]  # edit this list to run one model

TRAIN_FILENAME = "train_input_with_label.csv"
TEST_FILENAME = "test_input_with_label.csv"
OUT_ROOT = Path("Datasets/augmented_per_class_datasets")

ATTACK_TYPE_COL = "Attack Type"
LABEL_COL = "Label"
ATTACK_TOOL_COL = "Attack Tool"

RANDOM_SEED = 42


## Helpers


In [ ]:
def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)

def mode_or_none(series: pd.Series):
    s = series.dropna()
    if s.empty:
        return None
    mode = s.mode()
    if not mode.empty:
        return mode.iloc[0]
    return s.iloc[0]

def infer_fill_value(train_series: pd.Series):
    if pd.api.types.is_numeric_dtype(train_series):
        return 0
    val = mode_or_none(train_series)
    return val if val is not None else ""

def add_label_columns(synth_df: pd.DataFrame, train_df: pd.DataFrame, class_name: str) -> pd.DataFrame:
    out = synth_df.copy()
    if ATTACK_TYPE_COL in train_df.columns and ATTACK_TYPE_COL not in out.columns:
        out[ATTACK_TYPE_COL] = class_name
    if LABEL_COL in train_df.columns and LABEL_COL not in out.columns:
        out[LABEL_COL] = mode_or_none(train_df[LABEL_COL])
    if ATTACK_TOOL_COL in train_df.columns and ATTACK_TOOL_COL not in out.columns:
        out[ATTACK_TOOL_COL] = mode_or_none(train_df[ATTACK_TOOL_COL])
    return out

def align_to_train_columns(synth_df: pd.DataFrame, train_df: pd.DataFrame, class_name: str) -> pd.DataFrame:
    synth_df = add_label_columns(synth_df, train_df, class_name)

    missing_cols = [c for c in train_df.columns if c not in synth_df.columns]
    for col in missing_cols:
        synth_df[col] = infer_fill_value(train_df[col])

    extra_cols = [c for c in synth_df.columns if c not in train_df.columns]
    if extra_cols:
        synth_df = synth_df.drop(columns=extra_cols)

    print(f"[{class_name}] Added {len(missing_cols)} columns: {missing_cols}")
    print(f"[{class_name}] Dropped {len(extra_cols)} extra columns: {extra_cols}")
    return synth_df.reindex(columns=train_df.columns)

def align_df_to_reference_columns(df: pd.DataFrame, reference_columns) -> pd.DataFrame:
    out = df.copy()
    for col in reference_columns:
        if col not in out.columns:
            out[col] = np.nan
    extra_cols = [c for c in out.columns if c not in reference_columns]
    if extra_cols:
        out = out.drop(columns=extra_cols)
    return out.reindex(columns=reference_columns)

def shuffle_df(df: pd.DataFrame, seed: int) -> pd.DataFrame:
    return df.sample(frac=1.0, random_state=seed).reset_index(drop=True)

def augment_and_shuffle(train_df: pd.DataFrame, synth_df: pd.DataFrame, seed: int) -> pd.DataFrame:
    combined = pd.concat([train_df, synth_df], ignore_index=True)
    return shuffle_df(combined, seed)


## Merge augmented per-class train + synthetic sets and merge real test sets per model


In [ ]:
ensure_dir(OUT_ROOT)

for model_name in MODELS_TO_RUN:
    cfg = MODEL_CONFIGS.get(model_name)
    if not cfg:
        print(f"Skip model '{model_name}': not in MODEL_CONFIGS")
        continue

    synthetic_root = cfg["root"]
    synthetic_filename = cfg["synthetic_filename"]

    if not synthetic_root.exists():
        print(f"Skip model '{model_name}': missing {synthetic_root}")
        continue

    out_dir = OUT_ROOT / model_name
    ensure_dir(out_dir)

    combined_aug_parts = []
    combined_test_parts = []
    test_reference_columns = None

    for attack_dir in sorted([p for p in synthetic_root.iterdir() if p.is_dir()]):
        attack_name = attack_dir.name
        if attack_name.lower() == "balanced":
            continue

        synth_path = attack_dir / synthetic_filename
        train_path = attack_dir / TRAIN_FILENAME
        test_path = attack_dir / TEST_FILENAME

        # ---- Augmented train set ----
        if not synth_path.exists():
            print(f"[{model_name}] Skip augmentation for {attack_name}: missing {synthetic_filename}")
        elif not train_path.exists():
            print(f"[{model_name}] Skip augmentation for {attack_name}: missing {TRAIN_FILENAME}")
        else:
            print(f"[{model_name}] {attack_name} synth path: {synth_path}")
            print(f"[{model_name}] {attack_name} train path: {train_path}")

            train_df = pd.read_csv(train_path)
            synth_df = pd.read_csv(synth_path)
            synth_df = align_to_train_columns(synth_df, train_df, attack_name)

            aug_df = augment_and_shuffle(train_df, synth_df, RANDOM_SEED)
            out_path = out_dir / f"mixed_augmented_{attack_name}.csv"
            aug_df.to_csv(out_path, index=False)

            combined_aug_parts.append(train_df)
            combined_aug_parts.append(synth_df)

            print(
                f"[{model_name}] Saved {attack_name} -> {out_path} | "
                f"train {train_df.shape} + synth {synth_df.shape} = aug {aug_df.shape}"
            )

        # ---- Real test set merge for this model ----
        if not test_path.exists():
            print(f"[{model_name}] Skip test merge for {attack_name}: missing {TEST_FILENAME}")
        else:
            print(f"[{model_name}] {attack_name} test path: {test_path}")
            test_df = pd.read_csv(test_path)

            if test_reference_columns is None:
                test_reference_columns = list(test_df.columns)
            else:
                test_df = align_df_to_reference_columns(test_df, test_reference_columns)

            combined_test_parts.append(test_df)

    # Save merged augmented dataset for this model
    if combined_aug_parts:
        combined_aug_df = pd.concat(combined_aug_parts, ignore_index=True)
        combined_aug_df = shuffle_df(combined_aug_df, RANDOM_SEED)
        combined_aug_out = out_dir / "mixed_augmented_per_class_merged.csv"
        combined_aug_df.to_csv(combined_aug_out, index=False)
        print(f"[{model_name}] Saved combined augmented file -> {combined_aug_out} | shape {combined_aug_df.shape}")
    else:
        print(f"[{model_name}] No augmented outputs created; nothing to merge.")

    # Save merged real test dataset for this model
    if combined_test_parts:
        combined_test_df = pd.concat(combined_test_parts, ignore_index=True)
        combined_test_df = shuffle_df(combined_test_df, RANDOM_SEED)
        combined_test_out = out_dir / "test_input_with_label_merged.csv"
        combined_test_df.to_csv(combined_test_out, index=False)
        print(f"[{model_name}] Saved combined test file -> {combined_test_out} | shape {combined_test_df.shape}")
    else:
        print(f"[{model_name}] No test files found; nothing to merge.")
